## Import libraries

In [1]:
import numpy as np
import pandas as pd
import os

import warnings
warnings.filterwarnings("ignore")

## Load dataset

In [2]:
df = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Membuat salinan dataset

In [3]:
df_clean = df.copy()

print("Original Shape:", df.shape)
print("Working Copy Shape:", df_clean.shape)

Original Shape: (7043, 21)
Working Copy Shape: (7043, 21)


## Membersihkan whitespace

In [4]:
text_columns = df_clean.select_dtypes(
    include="object"
).columns

for column in text_columns:
    df_clean[column] = (
        df_clean[column]
        .str.strip()
    )

print("Whitespace cleaning completed.")

Whitespace cleaning completed.


## Mengubah TotalCharges menjadi numeric

In [5]:
df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"],
    errors="coerce"
)

print("TotalCharges Data Type:")
print(df_clean["TotalCharges"].dtype)

print(
    "Missing TotalCharges:",
    df_clean["TotalCharges"].isna().sum()
)

TotalCharges Data Type:
float64
Missing TotalCharges: 11


## Menangani missing values

In [6]:
missing_total = df_clean[
    df_clean["TotalCharges"].isna()
]

print("Missing TotalCharges:", len(missing_total))

print(
    "Tenure Distribution:",
    missing_total["tenure"].value_counts()
)

print(
    "Churn Distribution:",
    missing_total["Churn"].value_counts()
)

Missing TotalCharges: 11
Tenure Distribution: tenure
0    11
Name: count, dtype: int64
Churn Distribution: Churn
No    11
Name: count, dtype: int64


In [8]:
df_clean["TotalCharges"] = (
    df_clean["TotalCharges"].fillna(0)
)

print(
    "Missing After Cleaning:",
    df_clean["TotalCharges"].isna().sum()
)

Missing After Cleaning: 0


## Memeriksa duplikasi

In [9]:
print(
    "Duplicate Rows:",
    df_clean.duplicated().sum()
)

print(
    "Duplicate Customer IDs:",
    df_clean["customerID"].duplicated().sum()
)

Duplicate Rows: 0
Duplicate Customer IDs: 0


## Validasi tipe data

In [10]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [11]:
categorical_columns = df_clean.select_dtypes(
    include="object"
).columns

for column in categorical_columns:
    print(f"\n{column}")
    print(df_clean[column].value_counts(dropna=False))


customerID
customerID
3186-AJIEK    1
7590-VHVEG    1
5575-GNVDE    1
4501-VCPFK    1
6075-SLNIL    1
             ..
1452-KIOVK    1
6713-OKOMC    1
7892-POOKP    1
6388-TABGU    1
9763-GRSKD    1
Name: count, Length: 7043, dtype: int64

gender
gender
Male      3555
Female    3488
Name: count, dtype: int64

Partner
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Dependents
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

MultipleLines
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

OnlineSecurity
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup
OnlineBackup
No                     3088
Yes                    2429


In [12]:
validation = pd.DataFrame({
    "Column": df_clean.columns,
    "Data Type": df_clean.dtypes.values,
    "Missing Values": df_clean.isnull().sum().values,
    "Unique Values": df_clean.nunique().values
})

validation

,Column,Data Type,Missing Values,Unique Values
0,customerID,object,0,7043
1,gender,object,0,2
2,SeniorCitizen,int64,0,2
3,Partner,object,0,2
4,Dependents,object,0,2
5,tenure,int64,0,73
6,PhoneService,object,0,2
7,MultipleLines,object,0,3
8,InternetService,object,0,3
9,OnlineSecurity,object,0,3


In [13]:
output_dir = "../data/processed"

os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "telco_customer_churn_cleaned.csv"
)

df_clean.to_csv(
    output_path,
    index=False
)

print("Cleaned dataset successfully saved!")
print("Location:", output_path)

Cleaned dataset successfully saved!
Location: ../data/processed\telco_customer_churn_cleaned.csv
